In [1]:
# -*- coding: utf-8 -*-
"""
대안신용평가 모델 개발 프로젝트
변수 사용 가능성 검증 스크립트 (메인 A / 메인 B)

목적
----
실제 원본 데이터(AI-Hub 093-117)를 확보한 뒤, 계획서에서 사용하기로 한
변수들이 (1) 실제로 존재하는지, (2) 결측률이 어느 정도인지,
(3) 코드값이 스키마 문서와 일치하는지, (4) 목표변수(PERF1~4)의 클래스
불균형이 어느 정도인지, (5) 메인A-메인B 조인(ID/CUST_ID)이 가능한지를
한 번에 점검합니다.

사용법
------
1. RAW_A_PATH, RAW_B_PATH 에 실제 데이터 경로(csv/parquet 등)를 지정
2. python 변수_사용가능성_검증.py 실행
3. 콘솔 출력 + result 딕셔너리로 각 점검 결과 확인
"""

import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)

# ------------------------------------------------------------------
# 0. 데이터 경로 (실제 파일 확보 후 수정)
# ------------------------------------------------------------------
RAW_A_PATH = "../../data/9.개인_CB정보/202012_개인CB.csv"      # 메인 A: 9.개인CB정보
# RAW_B_PATH = "data/sheet11_통신카드CB결합정보.csv"  # 메인 B: 11.통신카드CB결합정보

# ------------------------------------------------------------------
# 1. 기획서 기준 "사용 예정 변수" 목록 (스키마 문서에서 추출)
#    - dict 형태: {필드영문명: (필드한글명, 코드여부Y/N, 유효값 리스트 or None)}
# ------------------------------------------------------------------
VARS_A = {
    "ID":         ("대체 Key",                         "N", None),
    "GENDER":     ("성별",                              "Y", ["1", "2"]),
    "AGE_BAND":   ("연령(10년 단위)",                    "Y", ["2","3","4","5","6","7","8","9"]),
    "SCORE":      ("KCB Score",                         "N", None),
    "SCORE_6M":   ("6개월전 KCB Score",                  "N", None),
    "PERF1":      ("12개월내 90일+ 연체경험",             "Y", ["0", "1"]),
    "PERF2":      ("12개월내 30일+ 연체경험",             "Y", ["0", "1"]),
    "PERF3":      ("3개월내 신규대출 발생",               "Y", ["0", "1"]),
    "PERF4":      ("6개월내 60일+ 연체경험",              "Y", ["0", "1"]),
    "AP0910001":  ("국민연금 성실납부 여부",              "Y", ["0", "1", "9"]),
    "AP0910002":  ("건강보험료 성실납부 여부",            "Y", ["0", "1", "9"]),
    "AL012G005":  ("3년내 자택주소 이력건수(구간화)",      "Y", ["-9","1","2","3","4","5","6","7"]),
    "AL012G011":  ("3년내 직장명 이력건수(구간화)",        "Y", ["-9","1","2","3","4","5"]),
    "AL012G019":  ("3년내 휴대폰번호 이력건수(구간화)",    "Y", ["-9","1","2","3","4","5"]),
}

# ------------------------------------------------------------------
# 2. 데이터 로드 (인코딩은 실제 파일에 맞게 조정: cp949 등 가능)
# ------------------------------------------------------------------
def load_data(path: str) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949")


# ------------------------------------------------------------------
# 3. 점검 함수들
# ------------------------------------------------------------------
def check_column_presence(df: pd.DataFrame, var_dict: dict, label: str) -> pd.DataFrame:
    """스키마상 존재해야 할 컬럼이 실제 데이터에 있는지 확인"""
    rows = []
    for col, (kor_name, is_code, _) in var_dict.items():
        rows.append({
            "필드영문명": col,
            "필드한글명": kor_name,
            "실제_존재여부": col in df.columns,
        })
    result = pd.DataFrame(rows)
    missing = result[~result["실제_존재여부"]]
    print(f"\n[{label}] 컬럼 존재 여부 점검")
    print(result.to_string(index=False))
    if len(missing):
        print(f"⚠ 존재하지 않는 컬럼 {len(missing)}개: {missing['필드영문명'].tolist()}")
    else:
        print("✅ 모든 예정 변수가 실제 데이터에 존재합니다.")
    return result


def check_missing_and_dtype(df: pd.DataFrame, var_dict: dict, label: str) -> pd.DataFrame:
    """결측률 및 데이터 타입 확인 (실제 존재하는 컬럼만 대상)"""
    cols = [c for c in var_dict if c in df.columns]
    stats = pd.DataFrame({
        "필드영문명": cols,
        "dtype": [df[c].dtype for c in cols],
        "결측률(%)": [round(df[c].isna().mean() * 100, 2) for c in cols],
        "고유값개수": [df[c].nunique(dropna=True) for c in cols],
    })
    print(f"\n[{label}] 결측률 / 타입 점검")
    print(stats.to_string(index=False))
    high_missing = stats[stats["결측률(%)"] > 30]
    if len(high_missing):
        print(f"⚠ 결측률 30% 초과 변수: {high_missing['필드영문명'].tolist()}")
    return stats


def check_valid_codes(df: pd.DataFrame, var_dict: dict, label: str):
    """코드형(Y) 변수의 실제 값이 스키마에 정의된 유효값과 일치하는지 확인"""
    print(f"\n[{label}] 코드값 유효성 점검")
    for col, (kor_name, is_code, valid_values) in var_dict.items():
        if is_code == "Y" and valid_values is not None and col in df.columns:
            actual_values = set(df[col].dropna().astype(str).unique())
            expected_values = set(valid_values)
            unexpected = actual_values - expected_values
            print(f"- {col} ({kor_name})")
            print(f"    실제값: {sorted(actual_values)}")
            if unexpected:
                print(f"    ⚠ 스키마에 없는 값 발견: {sorted(unexpected)}")
            else:
                print("    ✅ 스키마와 일치")


def check_perf_class_balance(df: pd.DataFrame, perf_cols=("PERF1", "PERF2", "PERF3", "PERF4")):
    """목표변수(PERF1~4) 클래스 불균형 점검 — 특히 SCORE=0(씬파일러) 구간"""
    print("\n[메인 A] PERF1~4 클래스 불균형 점검")
    for col in perf_cols:
        if col not in df.columns:
            continue
        vc = df[col].value_counts(dropna=False, normalize=True) * 100
        print(f"- {col} 분포(%): \n{vc.round(2)}")

    if "SCORE" in df.columns:
        thin = df[df["SCORE"] == 0]
        print(f"\nSCORE=0(씬파일러) 대상 건수: {len(thin):,} / 전체 {len(df):,} "
              f"({len(thin)/len(df)*100:.2f}%)")
        for col in perf_cols:
            if col in thin.columns:
                rate = thin[col].isna().mean() * 100
                print(f"  - 씬파일러 구간 {col} 결측률: {rate:.2f}%  "
                      f"(라벨이 없으면 지도학습 목표변수로 쓰기 어려움)")


def check_join_feasibility(df_a: pd.DataFrame, df_b: pd.DataFrame,
                            key_a: str = "ID", key_b: str = "CUST_ID"):
    """메인 A(ID)와 메인 B(CUST_ID) 조인 가능성 점검 — 기획서상 결합하지 않기로
    했지만, 교차검증(H6) 목적의 일부 표본 매칭 가능성 확인용"""
    print("\n[메인 A ↔ 메인 B] 키 조인 가능성 점검 (참고용)")
    if key_a not in df_a.columns or key_b not in df_b.columns:
        print("⚠ 키 컬럼이 존재하지 않아 점검 불가")
        return
    ids_a = set(df_a[key_a].astype(str).unique())
    ids_b = set(df_b[key_b].astype(str).unique())
    overlap = ids_a & ids_b
    print(f"메인 A 고유 ID 수: {len(ids_a):,}")
    print(f"메인 B 고유 ID 수: {len(ids_b):,}")
    print(f"교집합: {len(overlap):,} ({len(overlap)/max(len(ids_a),1)*100:.4f}% of A)")
    if len(overlap) == 0:
        print("→ 예상대로 두 데이터는 별개 모집단/식별체계로 보이며, "
              "개인 단위 결합은 불가능합니다. (기획서 구조와 일치)")


def check_age_definition_mismatch(df_a: pd.DataFrame, df_b: pd.DataFrame):
    """AGE_BAND(A, 10년 단위) vs AGE(B, 5년 단위) 정의 불일치 확인"""
    print("\n[메인 A vs 메인 B] 연령 구간 정의 비교")
    if "AGE_BAND" in df_a.columns:
        print("메인 A AGE_BAND 분포:")
        print(df_a["AGE_BAND"].value_counts().sort_index())
    if "AGE" in df_b.columns:
        print("메인 B AGE 분포:")
        print(df_b["AGE"].value_counts().sort_index())
    print("→ 두 코드체계가 다르므로, 청년(19~29세) 필터링 시 각 트랙에서 "
          "별도의 매핑 테이블을 정의해야 합니다.")


# ------------------------------------------------------------------
# 4. 실행부
# ------------------------------------------------------------------
if __name__ == "__main__":
    df_a = load_data(RAW_A_PATH)
    # df_b = load_data(RAW_B_PATH)

    result = {}
    result["A_존재여부"] = check_column_presence(df_a, VARS_A, "메인 A")
    result["A_결측_타입"] = check_missing_and_dtype(df_a, VARS_A, "메인 A")
    check_valid_codes(df_a, VARS_A, "메인 A")

    # result["B_존재여부"] = check_column_presence(df_b, VARS_B, "메인 B")
    # result["B_결측_타입"] = check_missing_and_dtype(df_b, VARS_B, "메인 B")
    # check_valid_codes(df_b, VARS_B, "메인 B")

    check_perf_class_balance(df_a)
    # check_join_feasibility(df_a, df_b)
    # check_age_definition_mismatch(df_a, df_b)

    print("\n모든 점검 완료. result 딕셔너리에 표 형태 결과가 저장되어 있습니다.")



[메인 A] 컬럼 존재 여부 점검
    필드영문명               필드한글명  실제_존재여부
       ID              대체 Key     True
   GENDER                  성별     True
 AGE_BAND          연령(10년 단위)     True
    SCORE           KCB Score     True
 SCORE_6M      6개월전 KCB Score     True
    PERF1     12개월내 90일+ 연체경험     True
    PERF2     12개월내 30일+ 연체경험     True
    PERF3        3개월내 신규대출 발생     True
    PERF4      6개월내 60일+ 연체경험     True
AP0910001        국민연금 성실납부 여부     True
AP0910002       건강보험료 성실납부 여부     True
AL012G005  3년내 자택주소 이력건수(구간화)     True
AL012G011   3년내 직장명 이력건수(구간화)     True
AL012G019 3년내 휴대폰번호 이력건수(구간화)     True
✅ 모든 예정 변수가 실제 데이터에 존재합니다.

[메인 A] 결측률 / 타입 점검
    필드영문명 dtype  결측률(%)   고유값개수
       ID   str     0.0 3129036
   GENDER int64     0.0       2
 AGE_BAND int64     0.0       9
    SCORE int64     0.0     661
 SCORE_6M int64     0.0     751
    PERF1 int64     0.0       2
    PERF2 int64     0.0       2
    PERF3 int64     0.0       2
    PERF4 int64     0.0       2
AP0910001 int64     0.0     